# OpenLoop KPI Analysis
This notebook analyzes the synthetic Promise Ledger dataset. All results are portfolio examples, not production outcomes.

In [ ]:
from pathlib import Path
import pandas as pd
DATA_DIR = Path('../data/csv')
promises = pd.read_csv(DATA_DIR / 'promises.csv', parse_dates=['created_ts','due_ts','completed_ts'])
departments = pd.read_csv(DATA_DIR / 'departments.csv')
conversations = pd.read_csv(DATA_DIR / 'conversations.csv')
notifications = pd.read_csv(DATA_DIR / 'notifications.csv')
promises.head()

## Core KPIs

In [ ]:
fulfilled = promises[promises['current_status'].eq('Fulfilled')].copy()
fulfilled['resolution_hours'] = (fulfilled['completed_ts'] - fulfilled['created_ts']).dt.total_seconds() / 3600
kpis = pd.Series({
    'Total promises': len(promises),
    'Fulfillment rate': promises['current_status'].eq('Fulfilled').mean(),
    'Overdue rate': promises['current_status'].eq('Overdue').mean(),
    'Average resolution hours': fulfilled['resolution_hours'].mean(),
    'Reassignment rate': promises['current_status'].eq('Reassigned').mean(),
    'Escalation rate': promises['current_status'].eq('Escalated').mean(),
    'SLA compliance among fulfilled': (fulfilled['completed_ts'] <= fulfilled['due_ts']).mean(),
    'Average AI confidence': promises['ai_confidence_score'].mean(),
})
kpis

## Department Performance

In [ ]:
department_summary = (
    promises.merge(departments[['department_id','department_name']], left_on='owner_department_id', right_on='department_id')
    .groupby('department_name', as_index=False)
    .agg(total_promises=('promise_id','count'),
         fulfilled=('current_status', lambda s: s.eq('Fulfilled').sum()),
         overdue=('current_status', lambda s: s.eq('Overdue').sum()),
         average_risk=('risk_score','mean'))
)
department_summary['fulfillment_rate'] = department_summary['fulfilled'] / department_summary['total_promises']
department_summary['overdue_rate'] = department_summary['overdue'] / department_summary['total_promises']
department_summary.sort_values('fulfillment_rate', ascending=False)

## Status Distribution

In [ ]:
promises['current_status'].value_counts().to_frame('promise_count')